# Semana 1: Fundamentos Lingüísticos y Primeros Pasos con NLP
## Notebook Conceptual (NB1) – Explorando el Lenguaje con Datos Dummy

**Propósito:** Introducir al estudiante en el campo del NLP, sus desafíos y las herramientas básicas de procesamiento. Se exploran los niveles del lenguaje y se realiza una primera toma de contacto con NLTK y spaCy.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Definir un pequeño corpus de oraciones y etiquetarlas manualmente con POS tags.
- Usar NLTK y spaCy para obtener automáticamente los POS tags y comparar resultados.
- Visualizar árboles de dependencia con spaCy.
- Identificar entidades nombradas (NER) en textos dummy.
- Discutir la ambigüedad léxica y estructural con ejemplos.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y descargamos los recursos lingüísticos requeridos.

In [ ]:
# Importamos librerías
import nltk
import spacy
from spacy import displacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Descargar recursos de NLTK
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('tagsets', quiet=True)

# Cargar modelo pequeño de spaCy para inglés
try:
    nlp = spacy.load('en_core_web_sm')
    print("Modelo de spaCy cargado correctamente.")
except:
    print("Descargando modelo de spaCy...")
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print("\nLibrerías importadas correctamente.")

---
## 1. Definición de un Corpus Dummy

Creamos un pequeño conjunto de oraciones inventadas que nos permitirán explorar conceptos lingüísticos. Incluimos ejemplos con ambigüedad léxica y estructural.

In [ ]:
# Definimos nuestro corpus de oraciones
corpus = [
    "The cat sits on the mat.",
    "The cats are sitting on the mats.",
    "The bank by the river is a great place to sit.",
    "I need to go to the bank to withdraw money.",
    "The old man the boats.",  # Oración famosa por su ambigüedad
    "Time flies like an arrow; fruit flies like a banana.",  # Ambigüedad léxica
    "Apple is looking to buy a U.K. startup for $1 billion."
]

print("Corpus definido:")
for i, sentence in enumerate(corpus):
    print(f"{i+1}. {sentence}")

---
## 2. Etiquetado Manual de Part-of-Speech (POS)

Antes de usar herramientas automáticas, vamos a etiquetar manualmente una oración para entender el concepto.

**Conjunto de etiquetas simplificado:**
- DET: determinante (the, a)
- NOUN: sustantivo (cat, bank, money)
- VERB: verbo (sits, need, flies)
- ADJ: adjetivo (old, great)
- ADP: preposición (on, by, to)
- PRON: pronombre (I)
- CONJ: conjunción (and, like)
- PUNCT: puntuación (.)

In [ ]:
# Oración a etiquetar manualmente
oracion_ejemplo = "The cat sits on the mat."
tokens = oracion_ejemplo.split()

# Etiquetado manual (simulado - el estudiante debería hacerlo realmente)
manual_tags = [('The', 'DET'), ('cat', 'NOUN'), ('sits', 'VERB'), 
               ('on', 'ADP'), ('the', 'DET'), ('mat', 'NOUN'), ('.', 'PUNCT')]

print("Etiquetado manual:")
for token, tag in manual_tags:
    print(f"{token:10} -> {tag}")

---
## 3. POS Tagging con NLTK

NLTK utiliza un etiquetador estadístico (Perceptron Tagger) que asigna etiquetas del Penn Treebank tagset.

In [ ]:
# Función para mostrar etiquetas NLTK
def pos_tag_nltk(sentence):
    tokens = nltk.word_tokenize(sentence)
    tagged = nltk.pos_tag(tokens)
    return tagged

print("=== POS Tagging con NLTK ===")
for sentence in corpus[:3]:  # primeras 3 oraciones
    print(f"\nOración: {sentence}")
    tagged = pos_tag_nltk(sentence)
    for word, tag in tagged:
        print(f"{word:15} -> {tag}")

## 4. POS Tagging con spaCy

spaCy utiliza modelos neuronales y asigna tanto etiquetas de POS detalladas (tag_) como simplificadas (pos_).

In [ ]:
def pos_tag_spacy(sentence):
    doc = nlp(sentence)
    return [(token.text, token.tag_, token.pos_) for token in doc]

print("=== POS Tagging con spaCy ===")
for sentence in corpus[:3]:
    print(f"\nOración: {sentence}")
    tagged = pos_tag_spacy(sentence)
    print(f"{'Palabra':15} {'Tag (detallado)':15} {'POS (simple)'}")
    print("-" * 50)
    for word, tag, pos in tagged:
        print(f"{word:15} {tag:15} {pos}")

### 4.1. Comparación NLTK vs spaCy

Observamos las diferencias en los tagsets y posibles variaciones en la asignación.

In [ ]:
sentence_test = "The cat sits on the mat."

nltk_tags = pos_tag_nltk(sentence_test)
spacy_tags = pos_tag_spacy(sentence_test)

print("Comparación de etiquetado:")
print(f"{'Palabra':10} {'NLTK':8} {'spaCy (detallado)':20} {'spaCy (simple)'}")
print("-" * 60)
for (word_nltk, tag_nltk), (word_spacy, tag_spacy, pos_spacy) in zip(nltk_tags, spacy_tags):
    print(f"{word_nltk:10} {tag_nltk:8} {tag_spacy:20} {pos_spacy}")

---
## 5. Visualización de Árboles de Dependencia

spaCy incluye un visualizador integrado (`displacy`) que muestra las relaciones de dependencia entre palabras.

In [ ]:
# Oración simple para visualizar
doc = nlp("The cat sits on the mat.")

# Visualizar dependencias
displacy.render(doc, style='dep', jupyter=True, options={'distance': 90})

In [ ]:
# Visualizar una oración ambigua
doc_amb = nlp("The old man the boats.")
displacy.render(doc_amb, style='dep', jupyter=True, options={'distance': 90})

In [ ]:
# Explicar algunas relaciones comunes
for token in doc:
    print(f"{token.text:10} -> {token.dep_:15} -> cabeza: {token.head.text}")

---
## 6. Reconocimiento de Entidades Nombradas (NER)

Identificamos entidades como personas, organizaciones, lugares, fechas, cantidades monetarias, etc.

In [ ]:
# Oración con varias entidades
ner_text = "Apple is looking to buy a U.K. startup for $1 billion."
doc_ner = nlp(ner_text)

# Mostrar entidades
print("Entidades detectadas por spaCy:")
for ent in doc_ner.ents:
    print(f"- {ent.text:20} -> {ent.label_:10} ({spacy.explain(ent.label_)})")

In [ ]:
# Visualización de entidades
displacy.render(doc_ner, style='ent', jupyter=True)

In [ ]:
# Probamos en más oraciones
print("=== NER en otras oraciones ===")
for sentence in corpus[2:4] + [corpus[-1]]:
    doc = nlp(sentence)
    print(f"\nOración: {sentence}")
    if doc.ents:
        for ent in doc.ents:
            print(f"  Entidad: {ent.text} ({ent.label_})")
    else:
        print("  No se detectaron entidades.")

---
## 7. Discusión sobre Ambigüedad

### 7.1. Ambigüedad Léxica

Una misma palabra puede tener diferentes significados según el contexto.

In [ ]:
amb_words = [
    ("bank", "I need to go to the bank to withdraw money."),
    ("bank", "The bank by the river is a great place to sit."),
    ("flies", "Time flies like an arrow."),
    ("flies", "Fruit flies like a banana.")
]

print("=== Ambigüedad léxica ===")
for word, context in amb_words:
    doc = nlp(context)
    print(f"\nPalabra: '{word}'")
    print(f"Contexto: {context}")
    # Buscamos el token correspondiente
    for token in doc:
        if token.text.lower() == word.lower():
            print(f"  POS tag: {token.tag_} ({token.pos_})")
            print(f"  Dependencia: {token.dep_}")

### 7.2. Ambigüedad Estructural

Una misma secuencia de palabras puede tener más de una interpretación sintáctica.

In [ ]:
amb_struct = [
    "The old man the boats.",
    "I saw the man with the telescope."
]

print("=== Ambigüedad estructural ===")
for sentence in amb_struct:
    print(f"\nOración: {sentence}")
    doc = nlp(sentence)
    print("Interpretación de spaCy (solo una posibilidad):")
    for token in doc:
        print(f"  {token.text:10} -> {token.dep_:15} -> {token.head.text}")

### 7.3. Discusión

**Preguntas para reflexionar:**

1. ¿Por qué el etiquetado POS automático puede equivocarse?
2. ¿Cómo afecta la ambigüedad léxica a tareas como la traducción automática?
3. ¿Qué información adicional necesitaría un sistema para resolver la ambigüedad de "The old man the boats"?
4. ¿Por qué es importante el contexto para el reconocimiento de entidades?

**Respuestas sugeridas:**

1. Los modelos estadísticos pueden fallar en contextos poco frecuentes o con palabras que tienen múltiples usos.
2. La traducción automática necesita desambiguar para elegir la palabra correcta en el idioma destino.
3. Necesita conocimiento de que "man" puede ser verbo (tripular) en este contexto, y que "old" es adjetivo.
4. Una misma palabra puede ser entidad o no según el contexto (ej. "Apple" como fruta vs como empresa).

---
## 8. Ejercicio para el Estudiante

Crea tu propia oración que contenga al menos:
- Una palabra con ambigüedad léxica
- Una posible entidad nombrada

Luego:
1. Etiquétala manualmente con POS tags (usando el conjunto simplificado).
2. Compáralo con los resultados de NLTK y spaCy.
3. Visualiza el árbol de dependencia.
4. ¿Las entidades fueron correctamente identificadas?

In [ ]:
# Espacio para que el estudiante escriba su código
# mi_oracion = "..."
# ...

---
## 9. Conclusiones

En este notebook hemos explorado los fundamentos del NLP:

✔️ **Corpus dummy**: Definimos oraciones con diferentes características lingüísticas.

✔️ **POS tagging manual vs automático**: Comparamos etiquetado manual con NLTK y spaCy.

✔️ **Árboles de dependencia**: Visualizamos la estructura sintáctica de las oraciones.

✔️ **NER**: Identificamos entidades nombradas en textos.

✔️ **Ambigüedad**: Discutimos ejemplos de ambigüedad léxica y estructural.

**Conceptos clave:**
- Los niveles del lenguaje (morfología, sintaxis, semántica, pragmática)
- La importancia del contexto para la desambiguación
- Herramientas básicas: NLTK y spaCy

En el próximo notebook (NB2) aplicaremos estos conceptos a un corpus real (Brown Corpus).

---
**Fin del Notebook Conceptual - Semana 1 (NLP)**